In [1]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)


# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

RANDOM_STATE = 42


# ---------------------------------------------------------
# Predictor and target definitions
# ---------------------------------------------------------

TARGET = "Machine failure"

NUMERIC_FEATURES = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

CATEGORICAL_FEATURES = [
    "Type",
]

FEATURES = (
    CATEGORICAL_FEATURES
    + NUMERIC_FEATURES
)


# ---------------------------------------------------------
# Locate dataset
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    if (PROJECT_ROOT.parent / "data").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "ai4i2020.csv"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATA_PATH.resolve()}"
    )


# ---------------------------------------------------------
# Load dataset
# ---------------------------------------------------------

df = pd.read_csv(DATA_PATH)

print(
    f"Dataset loaded: "
    f"{df.shape[0]:,} rows × "
    f"{df.shape[1]} columns"
)

Dataset loaded: 10,000 rows × 14 columns


In [2]:
# ---------------------------------------------------------
# Verify modelling variables
# ---------------------------------------------------------

required_columns = FEATURES + [TARGET]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: "
        f"{missing_columns}"
    )

if df[required_columns].isna().any().any():
    raise ValueError(
        "Missing values detected in modelling variables."
    )


# ---------------------------------------------------------
# Define X and y
# ---------------------------------------------------------

X = df[FEATURES].copy()
y = df[TARGET].copy()


# ---------------------------------------------------------
# Reproduce Stage 2/3 split
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Development set:        {len(X_train):,}")
print(f"Reserved final test set: {len(X_test):,}")

print(
    f"\nDevelopment failure rate: "
    f"{100 * y_train.mean():.2f}%"
)

print(
    f"Reserved test failure rate: "
    f"{100 * y_test.mean():.2f}%"
)

Development set:        8,000
Reserved final test set: 2,000

Development failure rate: 3.39%
Reserved test failure rate: 3.40%


In [3]:
# ---------------------------------------------------------
# Preprocessing
# ---------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            NUMERIC_FEATURES,
        ),
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
            ),
            CATEGORICAL_FEATURES,
        ),
    ]
)


# ---------------------------------------------------------
# Logistic Regression model
# ---------------------------------------------------------
# class_weight is intentionally left unchanged during
# this initial experiment.

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
)


# ---------------------------------------------------------
# Complete pipeline
# ---------------------------------------------------------

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("model", logistic_model),
    ]
)


# ---------------------------------------------------------
# Cross-validation
# ---------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print("Logistic Regression pipeline created.")
print("Cross-validation: 5-fold stratified")
print("Initial classification threshold: 0.50")

Logistic Regression pipeline created.
Cross-validation: 5-fold stratified
Initial classification threshold: 0.50


In [4]:
# ---------------------------------------------------------
# Out-of-fold failure probabilities
# ---------------------------------------------------------
# Each observation is predicted by a model that did not
# train on that observation.

failure_probability_oof = cross_val_predict(
    logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
)[:, 1]


# ---------------------------------------------------------
# Check probability range
# ---------------------------------------------------------

print(
    "Predicted failure-probability range: "
    f"{failure_probability_oof.min():.4f} "
    f"to {failure_probability_oof.max():.4f}"
)


# ---------------------------------------------------------
# Convert probability to class prediction
# ---------------------------------------------------------

CLASSIFICATION_THRESHOLD = 0.50

y_pred_oof = (
    failure_probability_oof
    >= CLASSIFICATION_THRESHOLD
).astype(int)


# ---------------------------------------------------------
# Classification performance
# ---------------------------------------------------------

classification_metrics = pd.DataFrame({
    "Score": {
        "Accuracy": accuracy_score(
            y_train,
            y_pred_oof,
        ),
        "Balanced Accuracy":
            balanced_accuracy_score(
                y_train,
                y_pred_oof,
            ),
        "Precision": precision_score(
            y_train,
            y_pred_oof,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_train,
            y_pred_oof,
            zero_division=0,
        ),
        "F1-score": f1_score(
            y_train,
            y_pred_oof,
            zero_division=0,
        ),
        "ROC-AUC": roc_auc_score(
            y_train,
            failure_probability_oof,
        ),
        "PR-AUC": average_precision_score(
            y_train,
            failure_probability_oof,
        ),
    }
})

print("\nCross-validated performance:")
display(
    classification_metrics.round(4)
)


# ---------------------------------------------------------
# Confusion matrix
# ---------------------------------------------------------

cm = confusion_matrix(
    y_train,
    y_pred_oof,
)

cm_df = pd.DataFrame(
    cm,
    index=[
        "Actual normal",
        "Actual failure",
    ],
    columns=[
        "Predicted normal",
        "Predicted failure",
    ],
)

print("\nCross-validated confusion matrix:")
display(cm_df)


# ---------------------------------------------------------
# Number of predicted failures
# ---------------------------------------------------------

predicted_failures = int(
    y_pred_oof.sum()
)

actual_failures = int(
    y_train.sum()
)

print(
    f"\nActual failures:    "
    f"{actual_failures:,}"
)

print(
    f"Predicted failures: "
    f"{predicted_failures:,}"
)

Predicted failure-probability range: 0.0001 to 0.9458

Cross-validated performance:


,Score
Accuracy,0.9704
Balanced Accuracy,0.5983
Precision,0.7297
Recall,0.1993
F1-score,0.3130
ROC-AUC,0.8932
PR-AUC,0.4589



Cross-validated confusion matrix:


,Predicted normal,Predicted failure
Actual normal,7709,20
Actual failure,217,54



Actual failures:    271
Predicted failures: 74
